## Decorator

---

A **decorator** wraps a function, adding behavior **around** it without modifying it and without changing its type.

Let:

- $X$ = the **input domain** — the argument type of the function being wrapped.
- $Y$ = the **output codomain** — its return type.
- $f : X \rightarrow Y$ = the **original function** — wrapped, never modified.
- $D$ = the **decorator** — a higher-order function: it takes $f$ and returns a new function of the **same** type $X \rightarrow Y$.

So the decorator's type is:

$$D : (X \rightarrow Y) \rightarrow (X \rightarrow Y)$$

To *define* $D$ we must say what the returned function $D(f)$ does when called. That added behavior is one function, the **augmentation** $h$, which is handed the original $f$ **and** the input $x$, and produces the output:

$$h : (X \rightarrow Y) \times X \rightarrow Y$$

The decorator is exactly "build the new function whose body is $h$":

$$\boxed{\,D(f) = x \mapsto h(f, x)\,}$$

Pointwise, for an input $x \in X$:

$$D(f)(x) \;=\; h(f, x)$$

The crucial point: **$f$ is passed *into* $h$ as a callable that $h$ may invoke** — zero times (caching hit, guard rejects), once (the usual case), or many times (retry). That freedom is what makes it a decorator rather than plain composition.

**Type check** — whatever $h$ does internally, it must return a $Y$, so the decorated function has the same signature as $f$:

$$\underbrace{x}_{X} \;\xrightarrow{\;h(\,\cdot\,,x)\;}\; \underbrace{h(f, x)}_{Y} \qquad\Rightarrow\qquad D(f) : X \rightarrow Y \;=\; \text{type of } f \;\checkmark$$

### The common shape: before / after hooks

Most decorators call $f$ **exactly once**, transforming the input first and the output after. Then $h$ factors into a *pre* map $b : X \rightarrow X$ and a *post* map $a : Y \rightarrow Y$:

$$h(f, x) = a\big(f(b(x))\big) \qquad\Longrightarrow\qquad D(f)(x) = a\big(f(b(x))\big)$$

Reading inside-out: transform input ($b$) → run $f$ → transform output ($a$). Special cases:

| Decorator does | $h(f,x)$ becomes | Example |
|---|---|---|
| post-process only | $a(f(x))$ &nbsp;($b=\mathrm{id}$) | "double the result" |
| pre-process only | $f(b(x))$ &nbsp;($a=\mathrm{id}$) | validate / log the input |
| skip $f$ sometimes | **cannot factor** — needs full $h(f,x)$ | caching, auth guard, retry |

The narrow formula $D(f)(x) = h(f(x))$ is just the **post-only** row. It can't express the bottom row, which is why the general definition keeps $f$ inside $h$.

### Stacking

Decorators return the same type they consume, so they compose — each layer wrapping the previous, in any order:

$$D_3\big(D_2(D_1(f))\big)(x)$$

### Map view

$$D : (X \rightarrow Y) \rightarrow (X \rightarrow Y), \qquad \text{client} \xrightarrow{\text{calls } D(f) \text{ as if it were } f} \text{(augmented) } f$$

**Conditions**

1. **Type preservation** — $D(f) : X \rightarrow Y$: same signature as $f$, so the client cannot tell it is talking to a decorator.
2. **Non-modification** — $f$ is never changed; each $D$ wraps around it without opening it.
3. **Composability** — decorators stack: $D_3(D_2(D_1(f)))$ applies all three layers.

> 🎁 Think of gift wrapping. The gift $f$ never changes. Each layer ($D$) adds something on the outside — tissue, then a box, then ribbon — each without opening the gift.


### Reading the definition in plain language

The formulas above, mapped onto the two exercises.

**The cast:**

| Symbol | Plain meaning | In `Milk(Coffee()).cost()` |
|---|---|---|
| $f$ | the **original** thing being wrapped | `Coffee()` |
| $X \rightarrow Y$ | $f$'s **shape** — takes an $X$, returns a $Y$ | `cost()`: takes nothing, returns a number |
| $D$ | the **wrapper** | `Milk` |
| $D(f)$ | the **wrapped** thing | `Milk(Coffee())` |
| $h$ | the **extra behavior** the wrapper adds | "$+\,0.50$ to the cost" |

**The type line.** $D : (X \rightarrow Y) \rightarrow (X \rightarrow Y)$ just says: *a decorator takes a function and gives back a function of the same shape.* In = `Coffee()` (a thing with `cost() → number`); out = `Milk(Coffee())` (**also** a thing with `cost() → number`). Same shape in, same shape out — which is exactly **why you can keep stacking**: the output is the same kind of thing the next wrapper accepts.

**The core formula.** $D(f)(x) = h(f, x)$ reads as:

> *"The wrapped thing, when called, runs the extra behavior $h$ — and $h$ is holding the original $f$, free to call it whenever it likes."*

For `Milk(Coffee()).cost()`:

$$\underbrace{\texttt{self.\_drink.cost()}}_{f}\; +\; \underbrace{0.50}_{h\text{'s extra}}$$

That **is** $h(f, x)$: $h$ = "add $0.50$", and it calls $f$ (`self._drink.cost()`) inside itself.

**Why $f$ lives *inside* $h$.** Putting $f$ inside $h$ (not just "$h$ after $f$") lets $h$ decide *whether* to call $f$:

- **Coffee** — $h$ calls $f$ once: inner cost, then $+\,0.50$. ($h(f(x))$, the simple case.)
- **Logging** — $h$ calls $f$ once: log, then `write`. (also simple)
- **Caching** — $h$ might **not** call $f$ at all: on a cache hit it returns the stored value and skips $f$. *This* is why the general form keeps $f$ inside $h$ rather than always running it first.

**The conditions, in one breath:**

- **Type preservation** — `Milk(Coffee())` still answers `cost()`/`description()` just like a plain `Coffee`. The customer can't tell it's wrapped.
- **Non-modification** — `Coffee` was never edited; `Milk` wrapped it from outside.
- **Composability** — because every layer keeps the same shape, `Vanilla(Milk(Sugar(Coffee())))` just works: each wrapper happily wraps the previous.


### Exercise 1 — Logging Decorator

---

**Scenario:** A `TextEditor` has `write(text)`. You want to log every call **without touching `TextEditor`**. The decorator wraps it, adding logging as the augmentation $h$.

**Your task:** Write a `LoggingDecorator` wrapping any text editor. It logs **before** every `write()` call, then delegates to $f$.

```python
editor = LoggingDecorator(TextEditor())   # D(f)
editor.write("Hello")
# [LOG] write() called with: Hello        <- h (the added behavior)
# Hello                                    <- f(x) (the original)
```

**Hints**

- The decorator holds `self._editor = editor` — this stores $f$. Its `write()` is the decorated call: print the log ($h$), **then** call `self._editor.write(text)` ($f$). Since the log runs *before* $f$, this is the **pre-hook** shape $D(f)(x) = f(b(x))$ where the "$b$" step is the logging side-effect.
- The decorator must expose the **same** interface as `TextEditor` — same method name `write`. This is the **type preservation** condition: $D(f) : X \rightarrow Y$, so the client can't tell it's talking to a decorator.


In [5]:
#--------------------------------
# Original (f) — you cannot change this

class TextEditor:
    def write(self, text):
        print(text)

#--------------------------------
# Decorator (D) — your task: log before delegating to f
import logging
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

class LoggingDecorator:
    def __init__(self, editor):
        self._editor = editor           # stores f

    def write(self, text):              # same interface as TextEditor (type preservation)
        # 1) log the call
        # 2) self._editor.write(text)                (f(x))
        logger.info(f"[LOG] write() called with: {text}")
        return self._editor.write(text)
        

#--------------------------------
editor = LoggingDecorator(TextEditor())   # D(f)
editor.write("Hello")
# expected:
# write() called with: Hello
# Hello


INFO:__main__:[LOG] write() called with: Hello


Hello


### Exercise 2 — Stacked Decorators (Coffee Shop)

---

**Scenario:** A `Coffee` has `cost()` and `description()`. Add-ons (`Milk`, `Sugar`, `Vanilla`) each increment the cost and extend the description — and can be **stacked in any order**.

**Your task:** Build add-on decorators that compose freely: $D_{\text{Vanilla}}\big(D_{\text{Milk}}(D_{\text{Sugar}}(f))\big)$.

```python
drink = Vanilla(Milk(Sugar(Coffee())))
print(drink.cost())         # sum of all layers
print(drink.description())  # Coffee, Sugar, Milk, Vanilla
```

**Hints**

- Each add-on stores the inner drink: `self._drink = drink`. Then `cost()` returns `self._drink.cost() + self.added_cost` — this is $h(f(x))$ where $h$ adds the increment ($f$ = the inner drink, untouched).
- Swap the wrapping order: the **total cost stays the same**, only the **description order changes**. This confirms the **composability** condition $D_3(D_2(D_1(f)))$.


In [ ]:
#--------------------------------
# Original (f) — the base drink, you cannot change this

class Coffee:
    def cost(self):
        return 2.0

    def description(self):
        return "Coffee"

#--------------------------------
# Add-on decorators (D) — each wraps a drink and adds to cost + description
# Every add-on exposes the SAME interface: cost() and description()  (type preservation)

class Sugar:
    added_cost = 0.25
    def __init__(self, drink):
        self._drink = drink                         # stores f (inner drink)
    def cost(self):
        return self._drink.cost() + self.added_cost # self._drink.cost() + self.added_cost
                                         
    def description(self):
        return self._drink.description() + ", Sugar" # self._drink.description() + ", Sugar"

class Milk:
    added_cost = 0.50
    def __init__(self, drink):
        self._drink = drink
    def cost(self):
        return self._drink.cost() + self.added_cost

    def description(self):
        return self._drink.description() + ", Milk"

class Vanilla:
    added_cost = 0.75
    def __init__(self, drink):
        self._drink = drink
    def cost(self):
        return self._drink.cost() + self.added_cost

    def description(self):
        return self._drink.description() + ", Vanilla"

#--------------------------------
drink = Vanilla(Milk(Sugar(Coffee())))              # D_Vanilla(D_Milk(D_Sugar(f)))
print(drink.cost())                                 # expected: 3.5
print(drink.description())                          # expected: Coffee, Sugar, Milk, Vanilla

# composability check — different order, SAME cost, different description order:
other = Sugar(Vanilla(Milk(Coffee())))
print(other.cost())                                 # expected: 3.5  (unchanged)
print(other.description())                          # expected: Coffee, Milk, Vanilla, Sugar


3.5
Coffee, Sugar, Milk, Vanilla
3.5
Coffee, Milk, Vanilla, Sugar
